# Step 2 — Clean the Investor Awareness data

**Project:** Investor Awareness Pulse: Financial Education Analytics  
**Focus event:** a generic Investor Awareness Workshop  
**Input:** `investor_awareness_pulse_raw_2500.csv`  
**Output:** `investor_awareness_pulse_clean.csv`

This notebook cleans the synthetic raw survey **column by column**. It does not overwrite the raw CSV. The data are fictional and for portfolio practice only.

## Cleaning decisions

- Keep the raw file unchanged; write a separate cleaned CSV.
- Normalize IDs, categories, and text; parse age as a nullable integer and dates as dates.
- For ambiguous slash dates, assume **day/month/year** (Indian date convention). Verify the original collection form if date accuracy matters.
- Use `Not reported` for missing categorical answers, and leave missing numeric/date values blank.
- Drop records with a missing respondent ID and keep the first record for repeated normalized IDs. The notebook reports how many rows this removes.
- Do not silently resolve conflicting awareness/attendance answers; flag them for review.

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
import re
from IPython.display import display


In [ ]:
file_path = r"C:\Users\Admin\Desktop\Workspace\Neo4j Project\Investor Awareness Pulse\Data Set\Unclean data\investor_awareness_pulse_raw_2500.csv"
output_path = r"C:\Users\Admin\Desktop\Workspace\Neo4j Project\Investor Awareness Pulse\Data Set\Clean data\investor_awareness_pulse_clean.csv"

RAW_PATH = Path(file_path)
CLEAN_PATH = Path(output_path)
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Raw file:", RAW_PATH)
print("Clean CSV will be saved to:", CLEAN_PATH)
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Check the raw CSV path: {RAW_PATH}")


## 1. Load and inspect the raw data

In [ ]:
df_raw = pd.read_csv(
    RAW_PATH,
    dtype=str,
    encoding="utf-8-sig",
    keep_default_na=False
)

print("Raw shape:", df_raw.shape)
display(df_raw.head(6))

In [ ]:
raw_profile = pd.DataFrame({
    "column": df_raw.columns,
    "blank_cells": [df_raw[c].astype(str).str.strip().eq("").sum() for c in df_raw.columns],
    "distinct_raw_values": [df_raw[c].nunique(dropna=False) for c in df_raw.columns],
    "example_values": [", ".join(df_raw[c].astype(str).drop_duplicates().head(5).tolist()) for c in df_raw.columns],
})
display(raw_profile)

## 2. Set up helpers and a clean output table

In [ ]:
def norm_key(value):
    """Lowercase and reduce punctuation/spacing for reliable category mapping."""
    if pd.isna(value):
        return ""
    text = str(value).casefold().strip()
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()

MISSING_KEYS = {"", "na", "n a", "n/a", "none", "null", "unknown", "not mentioned"}

def is_missing(value):
    return norm_key(value) in MISSING_KEYS

clean = pd.DataFrame(index=df_raw.index)

## 3. `respondent_id` — trim, uppercase, and deduplicate

In [ ]:
def clean_id(value):
    text = str(value).strip().upper()
    if is_missing(text):
        return pd.NA
    return re.sub(r"\s+", "", text)

clean["respondent_id"] = df_raw["respondent_id"].map(clean_id).astype("string")
missing_id_rows = int(clean["respondent_id"].isna().sum())
clean = clean.dropna(subset=["respondent_id"]).copy()
duplicate_id_rows = int(clean["respondent_id"].duplicated(keep="first").sum())
clean = clean.drop_duplicates(subset=["respondent_id"], keep="first").copy()

print("Rows with missing IDs removed:", missing_id_rows)
print("Repeated IDs removed (keeping first):", duplicate_id_rows)
print("Rows remaining:", len(clean))

## 4. `survey_date_raw` → `survey_date`

Dates are parsed into a consistent date value. For ambiguous slash dates, this notebook assumes day first; unparseable values remain missing for review.

In [ ]:
DATE_FORMATS = [
    "%Y-%m-%d", "%d/%m/%y", "%d/%m/%Y", "%m/%d/%y", "%m/%d/%Y",
    "%d-%b-%Y", "%d-%B-%Y", "%b %d, %Y", "%B %d, %Y",
    "%d %b %y", "%d %b %Y",
]

def parse_survey_date(value):
    text = str(value).strip()
    if is_missing(text) or norm_key(text) in {"not sure", "not remember", "sept 2026"}:
        return pd.NaT
    # Try Indian day-first slash formats before month-first formats.
    ordered_formats = ["%Y-%m-%d", "%d/%m/%y", "%d/%m/%Y"] + DATE_FORMATS[5:] + ["%m/%d/%y", "%m/%d/%Y"]
    for fmt in ordered_formats:
        try:
            return pd.Timestamp(datetime.strptime(text, fmt).date())
        except ValueError:
            pass
    return pd.NaT

clean["survey_date"] = pd.to_datetime(df_raw.loc[clean.index, "survey_date_raw"].map(parse_survey_date))
print("Unparsed/missing dates:", int(clean["survey_date"].isna().sum()))
display(pd.DataFrame({"raw": df_raw.loc[clean.index, "survey_date_raw"].head(10), "clean": clean["survey_date"].head(10)}))

## 5. `age_raw` → `age`

Extract the numeric age from entries such as `29 yrs` or `about 40`. Values outside 13–100 and non-numeric answers are left missing for review.

In [ ]:
def parse_age(value):
    text = str(value).strip()
    key = norm_key(text)
    if is_missing(text) or key.startswith("under "):
        return pd.NA
    match = re.search(r"\d{1,3}", text)
    if not match:
        return pd.NA
    age = int(match.group())
    return age if 13 <= age <= 100 else pd.NA

clean["age"] = pd.Series(df_raw.loc[clean.index, "age_raw"].map(parse_age), index=clean.index, dtype="Int64")
print("Missing/invalid ages:", int(clean["age"].isna().sum()))
display(pd.DataFrame({"raw": df_raw.loc[clean.index, "age_raw"].head(10), "clean": clean["age"].head(10)}))

## 6. `gender_raw` → `gender` and `education_raw` → `education`

In [ ]:
def clean_gender(value):
    key = norm_key(value)
    if is_missing(value) or key in {"no answer", "rather not say", "prefer not to say", "n a"}:
        return "Not reported"
    if key in {"f", "female", "woman", "femle"}:
        return "Female"
    if key in {"m", "male", "man"}:
        return "Male"
    if key in {"nb", "non binary", "nonbinary"}:
        return "Non-binary"
    if "prefer not" in key or "rather not" in key:
        return "Not reported"
    return "Other / review"

def clean_education(value):
    key = norm_key(value)
    if is_missing(value) or key in {"na", "n a", "not mentioned"}:
        return "Not reported"
    if key in {"other", "other education"}:
        return "Other"
    if any(term in key for term in ["secondary", "high school", "10th", "12th", "school"]):
        return "Secondary / high school"
    if any(term in key for term in ["diploma", "vocational", "iti"]):
        return "Diploma / vocational"
    if any(term in key for term in ["postgraduate", "post grad", "masters", "master", " pg "]):
        return "Postgraduate"
    if key == "pg" or key.startswith("pg "):
        return "Postgraduate"
    if any(term in key for term in ["undergraduate", "graduate", "bachelor", "ug"]):
        return "Undergraduate"
    return "Other / review"

clean["gender"] = df_raw.loc[clean.index, "gender_raw"].map(clean_gender)
clean["education"] = df_raw.loc[clean.index, "education_raw"].map(clean_education)
display(clean[["gender", "education"]].head(10))

## 7. `state_raw` → `state` and `area_type_raw` → `area_type`

In [ ]:
STATE_MAP = {
    "uttar pradesh": "Uttar Pradesh", "up": "Uttar Pradesh", "u p": "Uttar Pradesh",
    "delhi": "Delhi", "new delhi": "Delhi", "nct delhi": "Delhi",
    "maharashtra": "Maharashtra", "mh": "Maharashtra",
    "bihar": "Bihar", "karnataka": "Karnataka", "ka": "Karnataka",
    "west bengal": "West Bengal", "wb": "West Bengal",
    "rajasthan": "Rajasthan", "gujarat": "Gujarat",
    "tamil nadu": "Tamil Nadu", "tn": "Tamil Nadu",
    "madhya pradesh": "Madhya Pradesh", "mp": "Madhya Pradesh", "m p": "Madhya Pradesh",
    "telangana": "Telangana", "haryana": "Haryana", "kerala": "Kerala",
    "odisha": "Odisha", "orissa": "Odisha", "assam": "Assam",
    "jharkhand": "Jharkhand", "punjab": "Punjab",
    "uttarakhand": "Uttarakhand", "uttaranchal": "Uttarakhand",
    "chhattisgarh": "Chhattisgarh", "chattisgarh": "Chhattisgarh",
}

def clean_state(value):
    key = norm_key(value)
    if is_missing(value) or key in {"india", "not sure"}:
        return "Not reported"
    return STATE_MAP.get(key, "Needs review")

def clean_area(value):
    key = norm_key(value)
    if is_missing(value) or key in {"not sure", "village town"}:
        return "Not reported" if is_missing(value) else "Needs review"
    if any(term in key for term in ["metro", "metropolitan"]):
        return "Metro"
    if any(term in key for term in ["rural", "village"]):
        return "Rural"
    if any(term in key for term in ["urban", "town", "city"]):
        return "Urban"
    return "Needs review"

clean["state"] = df_raw.loc[clean.index, "state_raw"].map(clean_state)
clean["area_type"] = df_raw.loc[clean.index, "area_type_raw"].map(clean_area)
display(clean[["state", "area_type"]].head(10))

## 8. `investment_info_source_raw` → `investment_info_source`

In [ ]:
def clean_info_source(value):
    key = norm_key(value)
    if is_missing(value) or key in {"not remember", "dont remember", "do not remember"}:
        return "Not reported"
    if "youtube" in key or "you tube" in key:
        return "YouTube"
    if any(term in key for term in ["instagram", "facebook", "social", "whatsapp forward"]):
        return "Social media"
    if any(term in key for term in ["friend", "family", "relative"]):
        return "Family / friends"
    if "bank" in key or "financial institution" in key:
        return "Bank / financial institution"
    if any(term in key for term in ["tv", "news", "newspaper", "paper"]):
        return "TV / news"
    if any(term in key for term in ["college", "workplace", "office"]):
        return "College / workplace"
    if any(term in key for term in ["official finance", "government finance", "investor education", "official website"]):
        return "Official financial education source"
    if any(term in key for term in ["adviser", "advisor", "consultant"]):
        return "Financial adviser"
    if "other" in key:
        return "Other"
    return "Needs review"

clean["investment_info_source"] = df_raw.loc[clean.index, "investment_info_source_raw"].map(clean_info_source)
display(clean["investment_info_source"].value_counts(dropna=False).to_frame("respondents"))

## 9. `aware_of_event_raw` → `aware_of_event` and `attended_event_raw` → `attended_event`

In [ ]:
def clean_awareness(value):
    key = norm_key(value)
    if is_missing(value):
        return "Not reported"
    if key in {"maybe", "not sure", "no idea", "unsure", "dont know", "do not know"}:
        return "Not sure"
    if key in {"no", "n", "not yet", "never heard", "not aware"} or key.startswith("never heard") or key.startswith("not aware"):
        return "No"
    if key in {"yes", "y", "aware"} or key.startswith("yes ") or key.startswith("heard") or "know the workshop" in key:
        return "Yes"
    return "Needs review"

def clean_attendance(value):
    key = norm_key(value)
    if is_missing(value):
        return "Not reported"
    if key in {"maybe", "not sure", "no idea", "unsure", "dont know", "do not know"}:
        return "Not sure"
    if key in {"no", "n", "no never", "not yet"} or key.startswith("not attended") or key.startswith("no ") or key.startswith("never"):
        return "No"
    if key in {"yes", "y", "attended", "attended once", "participated"} or key.startswith("yes ") or "attend" in key or "participat" in key:
        return "Yes"
    return "Needs review"

clean["aware_of_event"] = df_raw.loc[clean.index, "aware_of_event_raw"].map(clean_awareness)
clean["attended_event"] = df_raw.loc[clean.index, "attended_event_raw"].map(clean_attendance)
# Flag contradictions rather than changing a respondent's answer automatically.
clean["response_consistency_flag"] = "OK"
conflict = (clean["aware_of_event"] == "No") & (clean["attended_event"] == "Yes")
missing_awareness_conflict = (clean["aware_of_event"] == "Not reported") & (clean["attended_event"] == "Yes")
clean.loc[conflict, "response_consistency_flag"] = "Review: attended but awareness marked No"
clean.loc[missing_awareness_conflict, "response_consistency_flag"] = "Review: attendance reported but awareness missing"
display(clean[["aware_of_event", "attended_event", "response_consistency_flag"]].head(10))

## 10. `preferred_channel_raw` → `preferred_channel` and clean free-text comments

In [ ]:
def clean_channel(value):
    key = norm_key(value)
    if is_missing(value):
        return "Not reported"
    if key in {"no preference", "not sure"}:
        return "No preference / not sure"
    if key in {"wa", "whatsapp"} or "whatsapp" in key:
        return "WhatsApp"
    if "youtube" in key or "you tube" in key:
        return "YouTube"
    if any(term in key for term in ["social", "instagram", "facebook"]):
        return "Social media"
    if any(term in key for term in ["person", "workshop", "offline", "session"]):
        return "In-person workshop"
    if any(term in key for term in ["email", "e mail"]):
        return "Email"
    if any(term in key for term in ["sms", "text message"]):
        return "SMS"
    if "bank" in key:
        return "Bank / branch"
    if any(term in key for term in ["newspaper", "paper"]):
        return "Newspaper"
    return "Needs review"

def clean_comment(value):
    if is_missing(value):
        return pd.NA
    text = re.sub(r"\s+", " ", str(value)).strip()
    return text if text else pd.NA

clean["preferred_channel"] = df_raw.loc[clean.index, "preferred_channel_raw"].map(clean_channel)
clean["respondent_comment"] = df_raw.loc[clean.index, "respondent_comment_raw"].map(clean_comment).astype("string")
display(clean[["preferred_channel", "respondent_comment"]].head(10))

## 11. Validate and save the cleaned CSV

In [ ]:
# Keep a useful, consistent column order for BI tools.
CLEAN_COLUMNS = [
    "respondent_id", "survey_date", "age", "gender", "education", "state", "area_type",
    "investment_info_source", "aware_of_event", "attended_event", "preferred_channel",
    "respondent_comment", "response_consistency_flag",
]
clean = clean[CLEAN_COLUMNS].reset_index(drop=True)

assert clean["respondent_id"].notna().all(), "Clean IDs should not be missing after the ID rule."
assert clean["respondent_id"].is_unique, "Respondent IDs should be unique after deduplication."

summary = pd.DataFrame({
    "metric": [
        "Raw input rows", "Rows removed: missing respondent ID", "Rows removed: duplicate respondent ID",
        "Clean output rows", "Dates missing/unparsed", "Age missing/invalid",
        "Responses flagged for consistency review",
    ],
    "value": [
        len(df_raw), missing_id_rows, duplicate_id_rows, len(clean),
        int(clean["survey_date"].isna().sum()), int(clean["age"].isna().sum()),
        int(clean["response_consistency_flag"].ne("OK").sum()),
    ],
})
display(summary)
display(clean.head(10))

clean.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print(f"Saved cleaned file: {CLEAN_PATH}")

## What to review before using the cleaned file

- Check the date convention against your original questionnaire. Ambiguous dates cannot be resolved with certainty from the raw text alone.
- Review `Needs review` categories and `response_consistency_flag` before final reporting.
- Keep both files: the original raw CSV and this cleaned CSV. Do not overwrite the raw source.